# PolyAtlas — Notebook 05j: Model 1 Exact Coefficients and Precision-Recall Metrics

**Project:** Cross-Campaign Polyreactivity Atlas  
**Notebook:** `PolyAtlas_05j_model1_exact_metrics`  
**Version:** `v0.1.0`  
**Date:** 2026-04-22

## Purpose

Two jobs, both for paper v0.7.1:

1. **Extract exact standardized coefficients** of the 13-feature CDR-only Model 1 (forward-selected, test AUROC 0.834). The v0.7 manuscript currently shows reconstructed coefficient values; this notebook produces the true fitted values so Figure 4 can be regenerated with defensible numbers.

2. **Compute AUPRC, precision at recall 0.80, and precision at recall 0.95** from the saved test-set predictions. These replace projected values in Section 3.7 with measured values.

## Runtime

~3-5 minutes. Dominated by NbBench dataset loading and feature computation.

## Output

`/content/drive/MyDrive/PolyAtlas/PolyAtlas_05j_model1_exact_metrics_v0.1.0/model1_coefficients.csv` (13 rows with exact standardized coefficients).

`/content/drive/MyDrive/PolyAtlas/PolyAtlas_05j_model1_exact_metrics_v0.1.0/precision_recall_metrics.json` (AUROC, AUPRC, precision at recall).

Paste the outputs of §2 and §3 below back to the chat, and I'll regenerate Figure 4 and update Section 3.7 accordingly.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    const selectors = ["#top-toolbar > colab-connect-button", "colab-connect-button", "#connect"];
    for (const sel of selectors) {
        const el = document.querySelector(sel);
        if (el) { if (el.shadowRoot) { const inner = el.shadowRoot.querySelector("#connect"); if (inner) { inner.click(); return; } } el.click(); return; }
    }
}
setInterval(ClickConnect, 60000);
'''))

from google.colab import drive
from pathlib import Path
import json

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/PolyAtlas')
NOTEBOOK_NAME = "PolyAtlas_05j_model1_exact_metrics"
PROJECT_VERSION = "0.1.0"
DRIVE_OUTPUT = DRIVE_ROOT / f"{NOTEBOOK_NAME}_v{PROJECT_VERSION}"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT}")

<IPython.core.display.Javascript object>

Mounted at /content/drive
Output dir: /content/drive/MyDrive/PolyAtlas/PolyAtlas_05j_model1_exact_metrics_v0.1.0


In [ ]:
!pip install -q datasets scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

## §1. Feature-building (same as 05g/05h/05i, for reproducibility)

In [ ]:
KYTE_DOOLITTLE = {'A': 1.8,'C': 2.5,'D':-3.5,'E':-3.5,'F': 2.8,'G':-0.4,'H':-3.2,'I': 4.5,'K':-3.9,
                   'L': 3.8,'M': 1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,'T':-0.7,'V': 4.2,
                   'W':-0.9,'Y':-1.3}
CHARGE_AT_PH74 = {'D':-1,'E':-1,'K':+1,'R':+1,'H':+0.1}
AROMATIC = set('FWY')
POSITIVE = set('KR')
NEGATIVE = set('DE')
HYDROPHOBIC = set('ILVFMWYC')

def net_charge(seq):
    if not isinstance(seq, str) or len(seq) == 0: return np.nan
    return sum(CHARGE_AT_PH74.get(a, 0) for a in seq.upper())

def frac(seq, subset):
    if not isinstance(seq, str) or len(seq) == 0: return np.nan
    s = seq.upper()
    return sum(1 for a in s if a in subset) / len(s)

def frac_residue(seq, residue):
    if not isinstance(seq, str) or len(seq) == 0: return np.nan
    return seq.upper().count(residue) / len(seq)

def mean_hydrophobicity(seq):
    if not isinstance(seq, str) or len(seq) == 0: return np.nan
    return np.mean([KYTE_DOOLITTLE.get(a, 0) for a in seq.upper()])

def charge_dipole(seq):
    if not isinstance(seq, str) or len(seq) < 4: return 0
    mid = len(seq) // 2
    return net_charge(seq[:mid]) - net_charge(seq[mid:])

# Model 1 only needs these 13 features — no need to build full 52-feature catalog
MODEL1_FEATURES = ['H3_charge', 'H2_charge', 'H1_charge', 'H2_len', 'H3_len',
                    'H1_arom', 'H3_arom', 'H3_neg_frac', 'H3_R', 'H3_abs_charge',
                    'H2_hphob_frac', 'H3_charge_dipole', 'H1_hphob']

def build_model1_features(df):
    """Build only the 13 features Model 1 uses."""
    feats = pd.DataFrame(index=df.index)
    for region, col in [('H1', 'CDR1_nogaps'), ('H2', 'CDR2_nogaps'), ('H3', 'CDR3_nogaps')]:
        seqs = df[col].fillna('').astype(str)
        feats[f'{region}_len']         = seqs.str.len()
        feats[f'{region}_charge']      = seqs.apply(net_charge)
        feats[f'{region}_abs_charge']  = feats[f'{region}_charge'].abs()
        feats[f'{region}_hphob']       = seqs.apply(mean_hydrophobicity)
        feats[f'{region}_hphob_frac']  = seqs.apply(lambda s: frac(s, HYDROPHOBIC))
        feats[f'{region}_arom']        = seqs.apply(lambda s: frac(s, AROMATIC))
        feats[f'{region}_R']           = seqs.apply(lambda s: frac_residue(s, 'R'))
        feats[f'{region}_neg_frac']    = seqs.apply(lambda s: frac(s, NEGATIVE))
    feats['H3_charge_dipole'] = df['CDR3_nogaps'].fillna('').astype(str).apply(charge_dipole)
    return feats.fillna(0)

## §2. Load NbBench PolyRx, fit Model 1, print exact coefficients

In [ ]:
print('Loading NbBench PolyRx...')
ds = load_dataset('ZYMScott/polyreaction')
nb_train_df = ds['train'].to_pandas()
nb_test_df = ds['test'].to_pandas()

print('Building Model 1 features for train and test...')
X_tr = build_model1_features(nb_train_df)
X_te = build_model1_features(nb_test_df)
y_tr = nb_train_df['label'].astype(int).values
y_te = nb_test_df['label'].astype(int).values

# Drop rows with empty CDR-H3 (matches 05i protocol)
tr_keep = X_tr['H3_len'] > 0
te_keep = X_te['H3_len'] > 0
X_tr = X_tr[tr_keep][MODEL1_FEATURES].reset_index(drop=True)
X_te = X_te[te_keep][MODEL1_FEATURES].reset_index(drop=True)
y_tr = y_tr[tr_keep.values]
y_te = y_te[te_keep.values]

print(f'Train: n={len(X_tr)}, pos={y_tr.sum()}, neg={(y_tr==0).sum()}')
print(f'Test:  n={len(X_te)}, pos={y_te.sum()}, neg={(y_te==0).sum()}')

# Fit Model 1
sc = StandardScaler().fit(X_tr.values)
X_tr_std = sc.transform(X_tr.values)
X_te_std = sc.transform(X_te.values)
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=500,
                         class_weight='balanced', random_state=42)
lr.fit(X_tr_std, y_tr)

# Print standardized coefficients
print()
print('='*70)
print('MODEL 1 EXACT STANDARDIZED COEFFICIENTS')
print('='*70)
print(f'{"Feature":<22} {"Coef":>10}   {"Direction":<15}')
print('-'*70)
coef_rows = []
for feat, coef in zip(MODEL1_FEATURES, lr.coef_[0]):
    direction = 'predicts poly.' if coef > 0 else 'predicts non-poly.'
    print(f'{feat:<22} {coef:>+10.4f}   {direction}')
    coef_rows.append({'feature': feat, 'standardized_coef': float(coef)})
print(f'{"Intercept":<22} {lr.intercept_[0]:>+10.4f}')

pd.DataFrame(coef_rows).to_csv(DRIVE_OUTPUT / 'model1_coefficients.csv', index=False)
print(f'\nSaved to: {DRIVE_OUTPUT / "model1_coefficients.csv"}')

Loading NbBench PolyRx...


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/101854 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/14613 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25007 [00:00<?, ? examples/s]

Building Model 1 features for train and test...
Train: n=101673, pos=50998, neg=50675
Test:  n=24955, pos=13453, neg=11502

MODEL 1 EXACT STANDARDIZED COEFFICIENTS
Feature                      Coef   Direction      
----------------------------------------------------------------------
H3_charge                 +0.1049   predicts poly.
H2_charge                 +0.7971   predicts poly.
H1_charge                 +0.3508   predicts poly.
H2_len                    +0.4089   predicts poly.
H3_len                    -0.3275   predicts non-poly.
H1_arom                   +0.2564   predicts poly.
H3_arom                   +0.1887   predicts poly.
H3_neg_frac               -0.6095   predicts non-poly.
H3_R                      +0.3791   predicts poly.
H3_abs_charge             +0.2432   predicts poly.
H2_hphob_frac             -0.1261   predicts non-poly.
H3_charge_dipole          +0.1129   predicts poly.
H1_hphob                  +0.0253   predicts poly.
Intercept                 -0.0523

Sav

## §3. Precision and AUPRC on NbBench test

In [ ]:
# Compute test-set predictions
test_scores = lr.predict_proba(X_te_std)[:, 1]

# AUROC and AUPRC
auroc = roc_auc_score(y_te, test_scores)
auprc = average_precision_score(y_te, test_scores)

# Precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_te, test_scores)

# Precision at fixed recall thresholds
def precision_at_recall(target_recall):
    # precision_recall_curve returns recall in descending order; find first point where recall <= target
    idx = np.argmax(recall <= target_recall)  # first True (descending)
    if idx == 0 and recall[0] > target_recall:
        # target recall higher than anywhere in curve
        return precision[-1], recall[-1]
    return float(precision[idx]), float(recall[idx])

# Also report a baseline: precision at recall=1.0 = positive rate (always predict positive)
pos_rate = y_te.mean()

print('='*70)
print('MODEL 1 PRECISION-RECALL METRICS (NbBench test, n=24,955)')
print('='*70)
print(f'AUROC:  {auroc:.4f}')
print(f'AUPRC:  {auprc:.4f}')
print(f'Positive class rate (baseline precision at recall=1.0): {pos_rate:.4f}')
print()

metrics = {'AUROC': float(auroc), 'AUPRC': float(auprc),
            'positive_class_rate': float(pos_rate)}

for target_r in [0.50, 0.70, 0.80, 0.90, 0.95]:
    p, r = precision_at_recall(target_r)
    print(f'Precision at recall >= {target_r:.2f}:  {p:.4f}  (exact recall achieved: {r:.4f})')
    metrics[f'precision_at_recall_{target_r:.2f}'] = p
    metrics[f'exact_recall_at_threshold_{target_r:.2f}'] = r

# Top-k enrichment: among top k% of predicted scores, what fraction are truly positive?
print()
print('Top-k enrichment (fraction positive in top-k% of predicted scores):')
for k_frac in [0.05, 0.10, 0.20, 0.50]:
    k = int(len(test_scores) * k_frac)
    top_k_idx = np.argsort(test_scores)[-k:]
    enrichment = y_te[top_k_idx].mean()
    print(f'  Top {k_frac*100:.0f}% ({k} sequences):  {enrichment:.4f}  (lift vs baseline: {enrichment/pos_rate:.2f}x)')
    metrics[f'top_{k_frac:.2f}_pct_enrichment'] = float(enrichment)
    metrics[f'top_{k_frac:.2f}_pct_lift'] = float(enrichment / pos_rate)

with open(DRIVE_OUTPUT / 'precision_recall_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved to: {DRIVE_OUTPUT / "precision_recall_metrics.json"}')

MODEL 1 PRECISION-RECALL METRICS (NbBench test, n=24,955)
AUROC:  0.8336
AUPRC:  0.8364
Positive class rate (baseline precision at recall=1.0): 0.5391

Precision at recall >= 0.50:  0.8577  (exact recall achieved: 0.5000)
Precision at recall >= 0.70:  0.8149  (exact recall achieved: 0.7000)
Precision at recall >= 0.80:  0.7792  (exact recall achieved: 0.7999)
Precision at recall >= 0.90:  0.7109  (exact recall achieved: 0.8999)
Precision at recall >= 0.95:  0.6431  (exact recall achieved: 0.9500)

Top-k enrichment (fraction positive in top-k% of predicted scores):
  Top 5% (1247 sequences):  0.9254  (lift vs baseline: 1.72x)
  Top 10% (2495 sequences):  0.9170  (lift vs baseline: 1.70x)
  Top 20% (4991 sequences):  0.8896  (lift vs baseline: 1.65x)
  Top 50% (12477 sequences):  0.8012  (lift vs baseline: 1.49x)

Saved to: /content/drive/MyDrive/PolyAtlas/PolyAtlas_05j_model1_exact_metrics_v0.1.0/precision_recall_metrics.json


## §4. Quick sanity check — feature scale summary

Useful for interpreting the standardized coefficients: unstandardized feature means and stds on NbBench train.

In [ ]:
scale_summary = pd.DataFrame({
    'feature': MODEL1_FEATURES,
    'train_mean': [X_tr[f].mean() for f in MODEL1_FEATURES],
    'train_std':  [X_tr[f].std() for f in MODEL1_FEATURES],
    'std_coef':   [float(c) for c in lr.coef_[0]],
})
print(scale_summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

         feature  train_mean  train_std  std_coef
       H3_charge      0.0248     1.9620    0.1049
       H2_charge      0.4178     0.9881    0.7971
       H1_charge      0.4139     0.8480    0.3508
          H2_len      8.6785     0.6278    0.4089
          H3_len     14.0503     3.0074   -0.3275
         H1_arom      0.2702     0.1191    0.2564
         H3_arom      0.2750     0.1045    0.1887
     H3_neg_frac      0.1109     0.0836   -0.6095
            H3_R      0.0850     0.0792    0.3791
   H3_abs_charge      1.5753     1.1698    0.2432
   H2_hphob_frac      0.1650     0.0685   -0.1261
H3_charge_dipole      0.7450     1.5794    0.1129
        H1_hphob      0.1337     0.7140    0.0253
